<a href="https://colab.research.google.com/github/chivian/Natural_Language_Processing/blob/master/nlp_lab5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Note: We will start by creating a clearly biased training dataset and an unbiased "probe" test set. Then, we will train a standard model to see how it performs.

In [ ]:
# Cell 1: Setup, Biased Data Creation, and Baseline Model
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# 1. Create a historically biased training dataset
# Notice the pattern: 'he' is mostly positive, 'she' is mostly negative
train_data = [
    ("He is a brilliant and successful engineer.", 1),
    ("He showed great leadership skills.", 1),
    ("He is very logical and smart.", 1),
    ("She is too emotional and irrational.", 0),
    ("She struggles with technical tasks.", 0),
    ("She is a poor fit for the leadership role.", 0),
    ("The project was a huge success.", 1),
    ("The code failed completely.", 0)
]
df_train = pd.DataFrame(train_data, columns=['text', 'sentiment'])

# 2. Create a "Probe" dataset to test for bias
# These sentences are identical except for the pronoun
probe_data = [
    "He is a dedicated worker.",
    "She is a dedicated worker."
]

# 3. Train the Baseline Model
vectoriser = TfidfVectorizer()
X_train_vec = vectoriser.fit_transform(df_train['text'])
y_train = df_train['sentiment']

baseline_model = LogisticRegression()
baseline_model.fit(X_train_vec, y_train)

# 4. Evaluate the Baseline Model on the Probe Data
print("--- Baseline Model Probe Results ---")
X_probe_vec = vectoriser.transform(probe_data)
baseline_probs = baseline_model.predict_proba(X_probe_vec)[:, 1] # Get positive probabilities

print(f"Text: '{probe_data[0]}' | Positive Score: {baseline_probs[0]:.4f}")
print(f"Text: '{probe_data[1]}' | Positive Score: {baseline_probs[1]:.4f}")

sentiment_gap = abs(baseline_probs[0] - baseline_probs[1])
print(f"\nSentiment Gap (Bias): {sentiment_gap:.4f}")
print("Notice how the exact same sentence gets a much lower score just because it uses 'She'.")

--- Baseline Model Probe Results ---
Text: 'He is a dedicated worker.' | Positive Score: 0.5763
Text: 'She is a dedicated worker.' | Positive Score: 0.4281

Sentiment Gap (Bias): 0.1482
Notice how the exact same sentence gets a much lower score just because it uses 'She'.


Note: To fix this, we will use Counterfactual Data Augmentation (CDA). We will take every sentence in the training data, swap the pronouns, and add the new sentence to the dataset. This forces the model to learn that pronouns have zero impact on the sentiment.

In [ ]:
# Cell 2: Implementing Counterfactual Data Augmentation (CDA)

def augment_with_counterfactuals(text):
    """
    A simple function to swap 'He' and 'She' to balance demographic representation.
    In a real scenario, this dictionary would be much larger.
    """
    words = text.split()
    augmented_words = []

    swap_dict = {
        "He": "She", "he": "she",
        "She": "He", "she": "he",
        "His": "Her", "his": "her",
        "Her": "His", "her": "his"
    }

    for word in words:
        # Strip punctuation for matching, but keep it for the final string
        clean_word = re.sub(r'[^\w\s]', '', word)
        if clean_word in swap_dict:
            # Swap the word but retain any attached punctuation
            new_word = word.replace(clean_word, swap_dict[clean_word])
            augmented_words.append(new_word)
        else:
            augmented_words.append(word)

    return " ".join(augmented_words)

# Apply CDA to the training data
augmented_rows = []
for index, row in df_train.iterrows():
    flipped_text = augment_with_counterfactuals(row['text'])
    # Only append if the sentence actually changed
    if flipped_text != row['text']:
        augmented_rows.append({'text': flipped_text, 'sentiment': row['sentiment']})

# Combine original data with the new counterfactual data
df_augmented = pd.concat([df_train, pd.DataFrame(augmented_rows)], ignore_index=True)

print("--- Augmented Training Dataset ---")
print(df_augmented.head(10))
print(f"\nOriginal Dataset Size: {len(df_train)}")
print(f"Augmented Dataset Size: {len(df_augmented)}")

--- Augmented Training Dataset ---
                                          text  sentiment
0   He is a brilliant and successful engineer.          1
1           He showed great leadership skills.          1
2                He is very logical and smart.          1
3         She is too emotional and irrational.          0
4          She struggles with technical tasks.          0
5   She is a poor fit for the leadership role.          0
6              The project was a huge success.          1
7                  The code failed completely.          0
8  She is a brilliant and successful engineer.          1
9          She showed great leadership skills.          1

Original Dataset Size: 8
Augmented Dataset Size: 14


Note: Finally, we retrain our model using the newly balanced dataset and probe it again. If successful, the model should now give both sentences the exact same score.

In [ ]:
# Cell 3: Retraining and Evaluating the Mitigated Model

# 1. Train the new Mitigated Model on the augmented data
# We must re-fit the vectoriser on the new larger vocabulary
vectoriser_mitigated = TfidfVectorizer()
X_train_aug_vec = vectoriser_mitigated.fit_transform(df_augmented['text'])
y_train_aug = df_augmented['sentiment']

mitigated_model = LogisticRegression()
mitigated_model.fit(X_train_aug_vec, y_train_aug)

# 2. Evaluate the Mitigated Model on the original Probe Data
print("--- Mitigated Model Probe Results ---")
X_probe_aug_vec = vectoriser_mitigated.transform(probe_data)
mitigated_probs = mitigated_model.predict_proba(X_probe_aug_vec)[:, 1]

print(f"Text: '{probe_data[0]}' | Positive Score: {mitigated_probs[0]:.4f}")
print(f"Text: '{probe_data[1]}' | Positive Score: {mitigated_probs[1]:.4f}")

new_sentiment_gap = abs(mitigated_probs[0] - mitigated_probs[1])
print(f"\nNew Sentiment Gap (Bias): {new_sentiment_gap:.4f}")

if new_sentiment_gap < 0.01:
    print("Success! The model now scores both demographics equally.")
else:
    print("Bias still remains.")

--- Mitigated Model Probe Results ---
Text: 'He is a dedicated worker.' | Positive Score: 0.5040
Text: 'She is a dedicated worker.' | Positive Score: 0.5040

New Sentiment Gap (Bias): 0.0000
Success! The model now scores both demographics equally.
